<a href="https://colab.research.google.com/github/burrows99/melanoma-classification/blob/master/colab_runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Melanoma Classification — Colab Runner

Clones (or updates) the repo, sets up `uv`, configures Kaggle credentials, runs 4 training experiments, then downloads plots and saved models.

In [1]:
REPO_URL = "https://github.com/burrows99/melanoma-classification.git"
REPO_DIR = "melanoma-classification"

# Always start from /content to avoid nested clones
%cd /content

# Remove stale clone if present, then clone fresh
!rm -rf {REPO_DIR}
!git clone {REPO_URL}

%cd {REPO_DIR}
print("Working directory: ", end="")
!pwd

/content
Cloning into 'melanoma-classification'...
remote: Enumerating objects: 168, done.
remote: Counting objects: 100% (159/159), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 168 (delta 71), reused 142 (delta 57), pack-reused 9 (from 1)
Receiving objects: 100% (168/168), 30.35 MiB | 24.43 MiB/s, done.
Resolving deltas: 100% (71/71), done.
/content/melanoma-classification
Working directory: /content/melanoma-classification


## Step 2 — Install uv and sync dependencies

In [2]:
import os

# Install uv (idempotent — safe to run every time)
!curl -LsSf https://astral.sh/uv/install.sh | sh

# Add uv to PATH for this session
uv_bin = os.path.expanduser("~/.local/bin")
os.environ["PATH"] = uv_bin + os.pathsep + os.environ.get("PATH", "")

# Sync all dependencies from pyproject.toml
!uv sync
print("Dependencies synced.")

downloading uv 0.11.7 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 102 packages in 0.99ms
Prepared 98 packages in 40.31s
Installed 98 packages in 297ms
 + albucore==0.0.24
 + albumentations==2.0.8
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + anyio==4.13.0
 + brotli==1.2.0
 + certifi==2026.2.25
 + charset-normalizer==3.4.7
 + click==8.3.2
 + contourpy==1.3.3
 + cuda-bindings==13.2.0
 + cuda-pathfinder==1.5.2
 + cuda-toolkit==13.0.2
 + cycler==0.12.1
 + fastapi==0.135.3
 + filelock==3.25.2
 + fonttools==4.62.1
 + fsspec==2026.3.0
 + grad-cam==1.5.5
 + gradio==6.12.0
 + gradio-client==2.4.1
 + groovy==0.1.2
 + h11==0.16.0
 + hf-gradio==0.3.0
 + hf-xet==1.4.3
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.10.1
 + idna==3.11
 + jinja2==3.1.6
 + joblib==1.5.3
 + kagglehub==1.0.0
 + kagglesdk==0.1.18
 + kiwisolver==1.5.0
 + markd

## Step 3 — Configure Kaggle credentials

Set your Kaggle username and API key below. You can find these at https://www.kaggle.com/settings → API → Create New Token.

In [3]:
import os, json, pathlib

KAGGLE_USERNAME = "mrburrows"  # <-- replace
KAGGLE_KEY      = "KGAT_7ebf0cae77a8cd59de30d0e1a51f7dfc"   # <-- replace

# Write ~/.kaggle/kaggle.json (required by kagglehub)
kaggle_cfg = pathlib.Path.home() / ".kaggle" / "kaggle.json"
kaggle_cfg.parent.mkdir(parents=True, exist_ok=True)
kaggle_cfg.write_text(json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}))
kaggle_cfg.chmod(0o600)

# Also export as environment variables (belt-and-suspenders)
os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"]      = KAGGLE_KEY

print("Kaggle credentials configured.")

Kaggle credentials configured.


## Step 4 — Run for 3 models


In [4]:
import os
# Colab sets MPLBACKEND=module://matplotlib_inline.backend_inline which is
# inherited by uv run subprocesses but not installed in the venv — force Agg.
os.environ["MPLBACKEND"] = "Agg"
print("MPLBACKEND set to Agg")


MPLBACKEND set to Agg


### 4.0 — Smoke Test (1 epoch, verifies full pipeline)


In [ ]:
!uv run main.py --train --epochs 1 --batch-size 32 --num-workers 0


Dataset split: 30118 train, 7530 validation samples.
Metadata preprocessor fitted. Output features: 14
Creating model 'efficientnet_b0' with 14 metadata features.
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100% 20.5M/20.5M [00:00<00:00, 207MB/s]

Epoch 1/1  |  LR: 0.0001
Training: 100% 941/941 [05:50<00:00,  2.68it/s]
Evaluating: 100% 236/236 [01:22<00:00,  2.85it/s]
Train: Loss=0.0154 | Acc=0.9004 | Recall=0.8930 | F1=0.7086
Val:   Loss=0.0117 | Acc=0.9263 | Recall=0.9295 | F1=0.7737
New best model saved: output/efficientnet_b0/weights/efficientnet_b0_Meta_LR0.0001_BS32_Ep1_BasicAug_best_ep1.pth  (Val F1: 0.7737)
Loading best model for final evaluation plots: output/efficientnet_b0/weights/efficientnet_b0_Meta_LR0.0001_BS32_Ep1_BasicAug_best_ep1.pth
Creating model 'efficientnet_b0' with 14 metadata features.
Evaluating: 100% 236/236 [01:23<00:00,  2.82it/s]
ROC c

### 4.1 — EfficientNet-B0

In [ ]:
!uv run main.py --train --architecture efficientnet_b0


Dataset not found in CWD — downloading from Kaggle (cached after first run)...
100% 1.01G/1.01G [01:05<00:00, 16.7MB/s]
Extracting files...
Dataset ready: 37648 images, CSV at dataset/train_concat.csv
Dataset split: 30118 train, 7530 validation samples.
Metadata preprocessor fitted. Output features: 14
Creating model 'efficientnet_b0' with 14 metadata features.
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100% 20.5M/20.5M [00:00<00:00, 233MB/s]

Epoch 1/20  |  LR: 0.0001
Training: 100% 941/941 [01:30<00:00, 10.43it/s]
Evaluating: 100% 236/236 [01:11<00:00,  3.32it/s]
Train: Loss=0.0153 | Acc=0.9016 | Recall=0.8905 | F1=0.7105
Val:   Loss=0.0107 | Acc=0.9247 | Recall=0.9334 | F1=0.7707
New best model saved: output/efficientnet_b0/weights/efficientnet_b0_Meta_LR0.0001_BS32_Ep20_BasicAug_best_ep1.pth  (Val F1: 0.7707)

Epoch 2/20  |  LR: 0.0001
Training: 100% 941/941 [

### 4.2 — DenseNet-121

In [5]:
!uv run main.py --train --architecture densenet121


07:28:02  __main__  INFO  Dataset not found in CWD — downloading from Kaggle (cached after first run)...
100% 1.01G/1.01G [00:28<00:00, 38.5MB/s]
Extracting files...
07:28:44  __main__  INFO  Dataset ready: 37648 images, CSV at dataset/train_concat.csv
Dataset split: 30118 train, 7530 validation samples.
Metadata preprocessor fitted. Output features: 14
07:28:56  model.metadata_melanoma_model  INFO  Creating model 'densenet121' with 14 metadata features.
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth
100% 30.8M/30.8M [00:00<00:00, 202MB/s]
07:28:57  train  INFO  Epoch 1/20  |  LR: 0.0001
Training: 100% 941/941 [01:36<00:00,  9.78it/s]
Evaluating: 100% 236/236 [02:26<00:00,  1.61it/s]
07:33:00  train  INFO  Train: Loss=0.0155 | Acc=0.9025 | Recall=0.8879 | F1=0.7118
07:33:00  train  INFO  Val:   Loss=0.0121 | Acc=0.9489 | Recall=0.8883 | F1=0.8249
07:33:00  train  INFO  New best model saved: outp

### 4.3 — ResNet-50

In [1]:
!uv run main.py --train --architecture resnet50


error: Failed to spawn: `main.py`
  Caused by: No such file or directory (os error 2)
